In [11]:
import random
import time
import pandas as pd
import numpy as np
from marg_substitution_extension import ClassificationDatabase, SalesDatabase, Product, Company, System
from SSELF_base import FootprintDatabase

# Step 1: Define classification info
classification_data = [
    {"class_code": "HS 2517", "class_name": "Crushed stone", "function": "Fills space", "unit": "m3"},
    {"class_code": "HS 2905", "class_name": "Alcohols", "function": "Chemical feedstock", "unit": "kg"},
    {"class_code": "HS 2618", "class_name": "Slag", "function": "Binder", "unit": "kg"},
]
classification_db = ClassificationDatabase(classification_data)

# Step 2: Initialize and populate last year's sales
last_year_sales = SalesDatabase(2023, classification_db)
base_function_outputs = {"HS 2517": 0.5, "HS 2905": 0.05, "HS 2618": 0.7}

# Add known secondary products
last_year_sales.add_sales(Product(12, "Crushed granite", "m3", "Company_11", "HS 2517", "secondary", 0.5, classification_db), 1000)
last_year_sales.add_sales(Product(14, "Glycerol", "kg", "Company_12", "HS 2905", "secondary", 0.05, classification_db), 2000)
last_year_sales.add_sales(Product(16, "Blast Furnace Slag", "kg", "Company_13", "HS 2618", "secondary", 0.7, classification_db), 3000)

# Add dummy products
codes = classification_db.data['class_code'].unique()
next_id = 17
extras = []
for code in codes:
    for _ in range(2):
        unit = classification_db.get_class_info(code)[2]
        extras.append({
            "id": next_id,
            "class_code": code,
            "sales_volume": np.random.uniform(500, 2000),
            "unit": unit,
            "function_output": base_function_outputs[code],
        })
        next_id += 1

# Mark 30% as constrained
dummy_df = pd.DataFrame(extras)
dummy_df['constrained'] = np.random.rand(len(dummy_df)) < 0.30
last_year_sales.data = pd.concat([last_year_sales.data, dummy_df], ignore_index=True)

#  Ensure all products in last_year_sales have a footprint in fp_2023
fp_2023 = FootprintDatabase(2023)
for pid in last_year_sales.data["id"].unique():
    fp_2023.report(pid, random.uniform(1, 100))

# Step 3: Build system
system = System(num_companies=13, num_products=19, classification_db=classification_db)

# Add co-products manually
co_products = [
    ("Crushed granite", "HS 2517", 0.025),
    ("Glycerol", "HS 2905", 0.05),
    ("Blast Furnace Slag", "HS 2618", 0.1),
]
for i, (name, code, fout) in enumerate(co_products):
    comp = system.companies[f"Company_{11 + i}"]
    pid = 17 + i
    co = Product(pid, name, "kg", comp, code, "secondary", fout, classification_db)
    comp.add_product(co)
    if comp.sales.empty:
        comp.sales = pd.DataFrame(columns=["Sales"])
    comp.sales.loc[pid] = [100]

# Step 4: Initialize footprint databases
fp_2024 = FootprintDatabase(2024)
for comp in system.companies.values():
    for prod in comp.products:
        fp_2024.report(prod.product_id, 0)

# Step 5: Run simulation
updates_completed = False
iteration = 0
start_time = time.time()

while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    any_updated = False

    for cname, comp in system.companies.items():
        if comp.check_update_needed(fp_2024, last_year_sales, fp_2023):
            print(f"{cname} was updated.")
            comp.update_footprint(fp_2024, fp_2023, last_year_sales)
            any_updated = True

    updates_completed = not any_updated

end_time = time.time()

# Step 6: Report
print(f"\n Updates completed in {iteration} iterations.")
print(f" Time taken: {end_time - start_time:.2f} seconds")

print("\n Final Footprints:")
for name, comp in system.companies.items():
    print(f"{name}: {comp.latest_update:.4f} kg CO2e/unit")

print("\n Update Count:")
total_updates = sum(c.num_updates for c in system.companies.values())
for name, c in system.companies.items():
    print(f"{name}: {c.num_updates} updates")
print(f"Total updates: {total_updates}")

print("\n Carbon balance check:")
for name, company in system.companies.items():
    company.check_carbon_balance(fp_2024)

# Constraint summary
print("\n=== Constraint Summary ===")
for cc in classification_db.data['class_code'].unique():
    df_cc = last_year_sales.data[last_year_sales.data['class_code'] == cc]
    total = len(df_cc)
    uncon = df_cc['constrained'].value_counts().get(False, 0)
    cons = df_cc['constrained'].value_counts().get(True, 0)
    print(f"Class {cc}: total={total}, unconstrained={uncon}, constrained={cons}")

# Detailed check per secondary
print("\n=== Detailed per‑secondary check ===")
for cname, comp in system.companies.items():
    for prod in comp.products:
        if prod.primary_secondary == "secondary":
            cc = prod.class_code
            print(f"\n{cname} → secondary product {prod.name} (class {cc}):")

            df_cc = last_year_sales.data[last_year_sales.data['class_code'] == cc]
            print("→ All candidates:")
            print(df_cc[['id', 'sales_volume', 'function_output', 'constrained']])

            if 'constrained' in df_cc.columns:
                df_uncon = df_cc[df_cc['constrained'] == False]
                print("→ Unconstrained candidates:")
                print(df_uncon[['id', 'sales_volume', 'function_output']])
            else:
                print("'constrained' column not found in data.")

            avg_fp = comp.get_average_footprint(cc, last_year_sales, fp_2023)
            print(f"Computed average footprint (unconstrained only): {avg_fp:.6f}")



--- Iteration 1 ---
Company_1 was updated.
Company_2 was updated.
Company_3 was updated.
Company_4 was updated.
Company_5 was updated.
Company_6 was updated.
Company_7 was updated.
Company_8 was updated.
Company_9 was updated.
Company_10 was updated.
Company_11 was updated.
Company_12 was updated.
Company_13 was updated.

--- Iteration 2 ---
Company_1 was updated.
Company_2 was updated.
Company_3 was updated.
Company_4 was updated.
Company_5 was updated.
Company_6 was updated.
Company_7 was updated.
Company_8 was updated.
Company_9 was updated.
Company_10 was updated.
Company_11 was updated.
Company_12 was updated.
Company_13 was updated.

--- Iteration 3 ---
Company_1 was updated.
Company_2 was updated.
Company_3 was updated.
Company_4 was updated.
Company_5 was updated.
Company_6 was updated.
Company_7 was updated.
Company_8 was updated.
Company_9 was updated.
Company_10 was updated.
Company_11 was updated.
Company_12 was updated.
Company_13 was updated.

--- Iteration 4 ---
Company

=== Constraint Summary ===
Class HS 2517: total=3, unconstrained=3, constrained=0
Class HS 2905: total=3, unconstrained=2, constrained=1
Class HS 2618: total=3, unconstrained=2, constrained=1

=== Detailed per‑secondary check ===

Company_11 → secondary product Crushed granite (class HS 2517):


,id,sales_volume,function_output,constrained
0,12,1000,0.5,False
3,17,682.896087,0.5,False
4,18,1901.708819,0.5,False


KeyError: "None of [Index([-1, -1, -1], dtype='object')] are in the [columns]"